In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

payments = spark.table(f"{catalog_name}.{silver_schema}.order_payments")
orders = spark.table(f"{catalog_name}.{silver_schema}.orders").select("order_id", "order_purchase_timestamp")

# payments ska date vete, pra e trashegon daten e blerjes nga orders permes order_id
fact_payments = (payments
    .join(orders, on="order_id", how="left")
    .withColumn("date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .select(
        "order_id", "payment_sequential", "payment_type",
        "payment_installments", "date_key", "payment_value"
    ))

(fact_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_payments"))
print(f"Wrote {fact_payments.count():,} payments")